In [6]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt

--2026-07-07 22:56:15--  https://raw.githubusercontent.com/karpathy/char-rnn/refs/heads/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-07-07 22:56:15 (21.2 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [15]:
dataset = open('input.txt', 'r', encoding='utf-8')
text = dataset.read()

In [24]:
print(len(text))

1115394


In [25]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [26]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars)) #the first element is actually '\n' and not ' ' like andrej said
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [27]:
encode_map = {ch:i for i,ch in enumerate(chars)} #maps characters to integers
decode_map = {i:ch for i,ch in enumerate(chars)} #maps integers back to characters

encode = lambda s: [encode_map[c] for c in s] #assigns each character of the string 's' to an int from the map 'encode_map'
decode = lambda l: [''.join(decode_map[c] for c in l)] #gets the corresponding character for every integer in the list created by the 'encode_map' for the string

In [28]:
print(encode('hello world'))
print(decode(encode('hello world'))) # first converting string to list of integers and then mapping back to characters joining them together

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
['hello world']


In [29]:
import torch
data = torch.tensor(encode(text), dtype=torch.long) # coverting all of the input.txt into encoded list of integers for each character. torch.long makes sure it is stored as integers.
print(data.shape,data.dtype)
print(data[:500])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [30]:
n = int(0.9* len(data))
train_data = data[:n]
val_data = data[n:]

In [32]:
torch.manual_seed(1337) # to get same numbers as andrej (helps in croschecking)

batch_size = 4
block_size = 8

def get_batch(split_type):
  data = train_data if split_type == 'train' else val_data
  ix = torch.randint(len(data)-block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x,y

xb, yb = get_batch('train')
print(xb.shape, '\n', xb)
print(yb.shape, '\n', yb)

torch.Size([4, 8]) 
 tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8]) 
 tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [52]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337) # to keep track with Andrej

class BigramLM(nn.Module):

  def __init__(self,vocab_size):
    super().__init__() # essentially to inherit the __init__() of the superclass (nn.Module)
    self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

  def forward(self, i, targets=None):
    logits = self.token_embedding_table(i) # (Batch, Time, Channel)
    if targets == None:
      loss = None
    else:
      B,T,C= logits.shape
      logits = logits.view(B*T,C)#3D to 2D compresses (4,8,65) to (32,65)
      targets = targets.view(B*T) #2D to 1D compresses (4,8) to (32,)
      loss = F.cross_entropy(logits,targets)
    return logits , loss

  def generate(self,i,max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(i) #runs forward() on each i in the (4,8) xb
      logits = logits[:,-1,:] #takes only the last character of each of the 4 rows so shape is now (4,65)
      probs = F.softmax(logits,dim=-1) #converts logits to probabilities
      # i_next = torch.multinomial(probs, num_samples = 1) #randomly picks one number (from the 65 available) weighted by the probabilities
      i_next = torch.argmax(probs, dim=-1, keepdim=True)  # instead of torch.multinomial(probs, num_samples=1)
      i = torch.cat((i,i_next), dim=1) # concatenates the 4 new i_next (shape is (4,1)) to the existing xb (shape of xb goes from 4,8 to 4,9 and so on every iteration)
    return i

m = BigramLM(vocab_size)
logits, loss = m(xb,yb)
print(loss)

tensor(4.8786, grad_fn=<NllLossBackward0>)


In [59]:
idx = torch.zeros((1,1), dtype = torch.long)
print(decode(m.generate(idx,100)[0].tolist()))

['\npxlgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcLIoiyhsrgCJAcL']
